# 펭귄 데이터로 배우는 기계학습 분류

설명 → 코드 완성 → 결과에서 이어지는 꼬리질문 → 자유 실험을 반복합니다.
원자료의 핵심 과정을 8개의 실습으로 골랐습니다. 약 45~60분이며 수업 상황에 따라 나누어 진행할 수 있습니다.

`___`를 직접 채우고 위에서 아래로 실행하세요. 코드 바로 뒤 질문은 방금 관찰한 결과를 근거로 답합니다.
서술형은 예시 답안과 스스로 비교하는 연습이며 자동 정오 판정을 하지 않습니다.


## 데이터와 첫 만남

`penguins_size.csv`는 남극 펭귄의 종·서식 섬·측정값을 담은 파일입니다.

| 열 이름 | 뜻 | 단위 |
|---|---|---|
| `species` | 펭귄의 종 | 없음 |
| `island` | 관측한 섬 | 없음 |
| `culmen_length_mm` | 부리 길이 | mm |
| `culmen_depth_mm` | 부리 깊이(두께) | mm |
| `flipper_length_mm` | 날개 길이 | mm |
| `body_mass_g` | 몸무게 | g |
| `sex` | 성별 표기 | 없음 |

### 질문 1 · 표의 한 행은 무엇을 기록한 걸까요?

아래 표에서 한 줄을 골라 읽어 보세요.

- 펭귄 한 마리의 관측값
- 한 섬에 사는 펭귄의 평균
- 한 종의 전체 개체 수

내 답: 

<details><summary>해설 보기</summary>한 행에 펭귄 한 마리의 종, 섬, 측정값, 성별이 함께 기록됩니다.</details>

### 질문 2 · 몸무게 3750과 날개 길이 181을 그대로 비교해도 될까요?

열 이름 끝의 g와 mm를 살펴보세요.

- 3750이 크므로 몸무게가 더 긴 값이다
- 단위와 측정 대상이 달라 숫자 크기만 비교할 수 없다
- 둘 다 펭귄의 측정값이므로 같은 단위다

내 답: 

<details><summary>해설 보기</summary>body_mass_g는 g, flipper_length_mm는 mm입니다. 숫자를 읽을 때 뜻과 단위도 함께 확인하세요.</details>

[실습 CSV](penguins_size.csv) · [데이터 출처](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data)


## 실행 준비
CSV를 이 노트북과 같은 폴더에 두고 아래 셀을 먼저 실행하세요. 브라우저 교재에서는 파일이 자동 준비됩니다.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["pandas", "matplotlib", "seaborn", "scikit-learn"])
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

## 01. 파일을 읽고 표의 크기 확인하기

이번에는 내장 예제가 아니라 실제 CSV 파일을 읽습니다. pandas의 read_csv()로 표를 df에 저장하고, shape와 head()로 크기와 앞부분을 확인하세요. 한 행은 펭귄 한 마리의 기록입니다.

**코드 완성 미션:** penguins_size.csv를 df로 읽고 표의 크기와 첫 5행을 출력하세요.

<details><summary>작성 힌트</summary>CSV를 읽는 pandas 함수는 read_csv입니다. 파일명은 따옴표 안에 정확히 입력합니다.</details>

In [ ]:
df = pd.___("penguins_size.csv")
print("표의 크기:", df.shape)
print(df.head())

### 질문 1

**글자로 된 값을 담은 열은 몇 개인가요?**

첫 5행에서 열별 값을 비교하세요.



내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>글자 열은 species, island, sex 3개입니다. 나머지 4개는 mm·g 단위의 측정값입니다. NaN은 값이 비었다는 표시일 뿐이고, 열이 글자 열인지 숫자 열인지는 바뀌지 않습니다. 5단계에서 숫자 열 4개만 입력으로 고르는 이유와 이어집니다.</details>

### 질문 2

**측정값으로 펭귄을 분류한다면, 무엇을 맞히도록 학습시킬까요?**

처음 보는 펭귄에도 쓸 모델을 생각해 보세요.

- island
- body_mass_g
- species
- culmen_length_mm

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>현장에서 기록하거나 잴 수 있는 값은 입력 후보이고, 아직 모르는 species가 정답(y)입니다. 예측할 때 알 수 없는 값을 입력에 넣으면 안 된다는 생각은 5단계 정답 누출 문항에서 다시 씁니다.</details>

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 02. 빈칸은 어느 열에 있을까?

표에 값이 없는 곳을 결측치라고 합니다. isna()로 빈 값을 찾고 sum()으로 열별 개수를 셉니다. 여러 열이 같은 행에서 비어 있을 수 있으므로, 열별 개수의 합과 삭제할 행 수는 다를 수 있습니다.

**코드 완성 미션:** 열별 결측치 개수를 missing에 저장하고 확인하세요.

<details><summary>작성 힌트</summary>빈 값을 찾는 메서드는 isna()입니다. 열마다 True의 개수를 합치면 결측치 개수가 나옵니다.</details>

In [ ]:
missing = df.___().sum()
print(missing)
print("한 곳이라도 비어 있는 행:", df.isna().any(axis=1).sum())

### 질문 1

**빈칸 18개가 왜 10행에서 나왔을까요?**

열별 결측치의 합과 마지막 출력을 비교하세요.

- 한 행에 빈칸이 여러 개 있으면, 열별로 셀 때 그 행이 여러 번 세어지기 때문이다.
- isna()가 숫자 열의 빈칸 중 일부를 찾지 못했기 때문이다.
- sex는 글자 열이라 '비어 있는 행'을 셀 때 빠지기 때문이다.
- 앞에서 일부 빈칸이 이미 지워진 뒤에 셌기 때문이다.

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>열별 개수는 빈칸 하나하나를 세고, 두 번째 출력은 빈칸이 하나라도 있는 펭귄(행)을 셉니다. 한 펭귄에게 빈칸이 5개 있으면 열별 합에서는 5번 세어지지만 행으로는 1마리입니다. 이 단계에서는 아직 아무것도 지우지 않았습니다.</details>

### 질문 2

**부리 길이가 빈 두 행의 sex는 어떤 상태일까요?**

빈칸이 있는 행은 10개이고, sex가 빈 행도 10개입니다.

- 둘 다 채워져 있다.
- 하나만 비어 있다.
- 지금 출력만으로는 추리할 수 없다.
- 둘 다 비어 있다.

자유 실험 셀에서 확인할 코드:

```python
df[df["culmen_length_mm"].isna()]
```

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>만약 그중 한 마리라도 sex가 채워져 있다면, sex가 빈 10마리에 그 한 마리가 더해져 비어 있는 행이 11개 이상이어야 합니다. 실제로는 10개이므로 빈칸이 있는 펭귄은 모두 sex도 빈 펭귄입니다. 자유 실험에서 df[df["culmen_length_mm"].isna()]를 실행하면 두 행(번호 3, 339)의 sex가 모두 NaN입니다. 따라서 10행은 '측정값과 sex가 모두 빈 2행'과 'sex만 빈 8행'으로 나뉩니다.</details>

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 03. 빈 값과 잘못 적힌 값 정리하기

이번 실습에서는 결측치가 있는 행을 제거합니다. 이어서 sex가 MALE 또는 FEMALE인 행만 남깁니다. 점(.)은 비어 있는 값이 아니라 문자이므로 dropna()만으로 없어지지 않습니다. 원본 df는 두고 clean을 만들어 비교합니다.

**코드 완성 미션:** 결측치를 제거하고 성별이 MALE/FEMALE인 기록만 clean에 남기세요.

<details><summary>작성 힌트</summary>결측치가 있는 행을 제거하는 메서드는 dropna()입니다. 이후 조건 필터가 점(.) 기록을 제외합니다.</details>

In [ ]:
clean = df.___().copy()
print("빈 행 제거 후:", len(clean))
print("성별 값:", clean["sex"].unique())
clean = clean[clean["sex"].isin(["MALE", "FEMALE"])].copy()
print("최종 기록 수:", len(clean))

### 질문 1

**dropna() 뒤에도 sex의 '.'이 남은 이유는 무엇일까요?**

성별 값에서 'MALE', 'FEMALE', '.'을 확인하세요.

- dropna()는 숫자 열만 검사하고, 글자 열인 sex는 검사하지 않기 때문이다.
- '.'은 빈칸이 아니라 누군가 실제로 입력한 글자 하나이기 때문이다.
- copy()가 지워진 행을 다시 복사해 왔기 때문이다.

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>dropna()는 NaN(값 없음)만 지웁니다. '.'은 글자이므로 '값이 있다'로 취급됩니다. 첫 번째 보기가 틀린 근거는 2단계에 있습니다. sex만 빈 8행도 dropna()로 지워졌으므로 sex 열도 검사한 것입니다. 값이 비었는지와 값이 올바른지는 따로 확인해야 하므로 isin() 필터를 한 번 더 씁니다.</details>

### 질문 2

**측정값이 하나라도 빈 행은 몇 개인가요?**

정리 전 df에서 측정값 4개 열만 기준으로 세어 보세요.



자유 실험 셀에서 확인할 코드:

```python
print(df[["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]].isna().any(axis=1).sum())
```

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>2행입니다. 이번 교재는 원자료 노트북처럼 sex까지 정리해 10행과 '.' 1행을 지우고 333행을 씁니다. 그런데 모델 입력은 측정값 4개뿐이므로, 측정값이 빈 행만 지우면 342행을 쓸 수도 있습니다. 어느 열을 기준으로 정리할지는 무엇을 분석하느냐에 따라 정하는 선택입니다. 이번 교재는 sex까지 정리된 표를 쓰는 쪽을 골랐습니다.</details>

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 04. 산점도로 특징 비교하기

원자료의 여러 그래프 중 산점도 하나를 골랐습니다. 가로축은 날개 길이(mm), 세로축은 몸무게(g), 색과 점 모양은 종입니다. 두 측정값만으로 종이 완전히 나뉘는지 관찰하세요. 관찰되는 관계만으로 원인과 결과를 단정하지는 않습니다.

**코드 완성 미션:** clean으로 종별 날개 길이와 몸무게의 산점도를 그리세요.

<details><summary>작성 힌트</summary>data에는 결측치와 성별 표기를 정리한 표 clean을 전달하세요.</details>

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=___, x="flipper_length_mm", y="body_mass_g", hue="species", style="species")
plt.title("Penguin measurements (clean records)")
plt.xlabel("Flipper length (mm)")
plt.ylabel("Body mass (g)")
plt.tight_layout()
plt.show()

### 질문 1

**이 그림에서 가장 구분하기 어려운 두 종은 무엇일까요?**

점의 색과 모양을 비교하세요.

- Adelie와 Chinstrap
- Adelie와 Gentoo
- Chinstrap과 Gentoo
- 세 종이 모두 비슷하게 섞여 있다

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>Gentoo는 날개가 길고 무거운 오른쪽 위에 대체로 따로 모입니다. Adelie(날개 172~210mm)와 Chinstrap(178~212mm)은 날개 길이 범위와 몸무게 평균(약 3,710g과 3,730g)이 거의 같아서 점이 많이 섞입니다.</details>

### 질문 2

**가로축을 부리 길이로 바꾸면 두 종이 더 잘 구분되나요? 그림을 근거로 설명하세요.**

Adelie와 Chinstrap을 비교하세요. 실험 코드로 축을 바꿔 그려 봅니다.



자유 실험 셀에서 확인할 코드:

```python
plt.figure(figsize=(7, 4))
sns.scatterplot(data=clean, x="culmen_length_mm", y="body_mass_g", hue="species", style="species")
plt.title("Penguins: culmen length and body mass")
plt.xlabel("Culmen length (mm)")
plt.ylabel("Body mass (g)")
plt.tight_layout()
plt.show()
```

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>예시: 쉬워졌다. 부리 길이 축에서는 Adelie가 대부분 왼쪽(약 32~46mm)에, Chinstrap은 오른쪽(약 41~58mm)에 모여 날개 길이 그림보다 덜 섞인다. 다만 41~46mm 근처에는 두 종이 섞인 점이 남아 있다.
덧붙임: 이번 그림에서는 Chinstrap과 Gentoo의 부리 길이가 비슷하지만 몸무게로는 대체로 나뉩니다. 한 측정값으로 모든 종을 나누기 어렵기 때문에 5단계에서 측정값 4개를 함께 씁니다.</details>

자기점검:
- [ ] 두 종이 그림의 어느 쪽(왼쪽·오른쪽, 위·아래)에 모였는지 위치로 적었다.
- [ ] '완전히 나뉜다'고 쓰지 않고 남은 겹침도 확인했다.
- [ ] 원래 그림(날개 길이)과 비교하는 말이 들어 있다.

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 05. 입력과 정답 나누기

부리 길이·부리 깊이·날개 길이·몸무게 4개를 입력 X로, species를 정답 y로 정합니다. 원자료의 MinMax 정규화는 이번 결정트리 실습에서 생략합니다. 수치의 크기를 맞추는 과정과 데이터 정리를 혼동하지 않도록 합니다.

**코드 완성 미션:** 수치 특성 4개를 X, 종을 y로 지정하세요.

<details><summary>작성 힌트</summary>X에는 열 이름 목록 features를 사용하고, y에는 맞혀야 할 열 이름 species를 넣습니다.</details>

In [ ]:
features = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
X = clean[___]
y = clean["___"]
print("입력 모양:", X.shape)
print("정답 종류:", y.unique())

### 질문 1

**몸무게 열을 빼면 행 수는 어떻게 될까요? 숫자로 예측하세요.**

열 3개만 선택합니다. 답한 뒤 실험 코드로 확인하세요.



자유 실험 셀에서 확인할 코드:

```python
clean[["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm"]].shape
```

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>333입니다. 열을 몇 개 고르든 펭귄 수(행)는 그대로이고 두 번째 숫자만 4에서 3으로 바뀝니다. X의 행 수와 y의 길이가 같아야 펭귄 한 마리의 입력과 정답이 짝을 이룹니다.</details>

### 질문 2

**이 모델을 새 펭귄에게 쓰기 어려운 이유는 무엇일까요?**

species를 숫자로 바꿔 X에 넣었더니 평가 정확도가 1.0이 나왔습니다.

- 입력이 5개로 늘어 계산이 느려지는 것이 가장 큰 문제다.
- 정확도 1.0은 펭귄의 특징을 완벽히 배웠다는 뜻이므로 문제가 없다.
- species를 숫자로 바꾸면서 종 이름이 사라진 것이 문제다.
- 새 펭귄의 species는 아직 모르는 값이라 그 입력 칸을 채울 수 없다.

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>맞혀야 할 정답을 입력에 넣는 것을 정답 누출이라고 합니다. 정확도 1.0은 측정값에서 규칙을 배운 결과가 아니라, 입력에 들어 있는 정답을 읽은 결과입니다. 1단계의 탐사대 상황처럼 실제로 예측할 때는 species를 모릅니다. 숫자로 바꾼 이유는 결정트리가 글자 열을 그대로 입력받지 못하기 때문입니다.</details>

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 06. 학습용과 평가용 나누기

전체의 약 70%로 학습하고 30%로 평가합니다. stratify=y는 각 종의 비율을 가급적 유지하며 나누도록 하고, random_state=42는 같은 결과를 다시 확인하기 위한 설정입니다.

**코드 완성 미션:** test_size=0.3으로 나누고 학습용·평가용 개수를 확인하세요.

<details><summary>작성 힌트</summary>30%는 소수 0.3입니다. test_size는 평가 데이터의 비율을 지정합니다.</details>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=___, stratify=y, random_state=42
)
print("학습용:", len(X_train), "평가용:", len(X_test))

### 질문 1

**평가용 데이터를 따로 남기는 이유는 무엇일까요?**

학습용 233마리와 평가용 100마리의 역할을 생각하세요.

- 학습할 데이터가 줄어야 학습 시간이 짧아지기 때문이다.
- 평가용 정확도가 학습용보다 높게 나오게 하기 위해서다.
- 학습 때 보지 못한 펭귄도 잘 맞히는지 확인하기 위해서다.
- 나누어 두면 결측치가 다시 섞이지 않기 때문이다.

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>연습한 문제를 그대로 시험에 내면 실력을 알 수 없습니다. 평가용 데이터는 모델이 처음 보는 펭귄 역할을 합니다. 그래서 7단계에서 fit()에는 X_train과 y_train만 넣고, X_test는 평가할 때만 씁니다.</details>

### 질문 2

**평가용 100마리 중 Chinstrap은 약 몇 마리일까요?**

333마리 중 Chinstrap은 68마리입니다. 종별 비율을 유지해 나눕니다.

- 약 7마리
- 약 20마리
- 약 30마리
- 68마리 모두

자유 실험 셀에서 확인할 코드:

```python
y_test.value_counts()
```

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>실제로는 Chinstrap 20마리(Adelie 44, Gentoo 36)입니다. 68마리의 30%가 약 20마리이고, 100마리의 약 20%로 생각해도 됩니다. stratify를 빼고 random_state=42로 나누면 18마리가 되고, random_state를 바꾸면 16마리나 24마리처럼 달라지기도 합니다. stratify는 종의 비율을 맞추고, random_state는 같은 분할을 다시 만듭니다. 두 설정은 하는 일이 다릅니다.</details>

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 07. 결정트리 학습하고 평가하기

학습용 데이터만 fit()에 전달합니다. 학습용 정확도와 평가용 정확도를 나란히 보고, 단순히 학습 문제를 잘 외운 것인지 살펴봅니다. 처음에는 나무의 깊이를 3으로 제한합니다.

**코드 완성 미션:** 결정트리를 학습하고 평가용 예측값 pred와 정확도 score를 구하세요.

<details><summary>작성 힌트</summary>학습은 fit(), 평가용 실제 정답은 y_test입니다. X_test와 y_test를 학습에 쓰지 않습니다.</details>

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.___(X_train, y_train)
pred = model.predict(X_test)
score = accuracy_score(___, pred)
print("학습용 정확도:", model.score(X_train, y_train))
print("평가용 정확도:", score)

### 질문 1

**평가 정확도 0.94는 무엇을 뜻할까요?**

평가용 펭귄이 100마리일 때를 생각하세요.

- 학습용 233마리 중 94%를 맞혔다.
- 모델이 94마리의 펭귄으로 학습했다.
- 평가용 100마리 중 94마리의 종을 맞혔다.
- 앞으로 만날 펭귄도 정확히 94%를 맞힐 것이다.

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>평가용 정확도는 평가용 100마리 중 맞힌 비율입니다. 학습용 정확도 약 0.970은 학습용 233마리 중 226마리를 맞혔다는 뜻입니다. 이 값은 한 번 나눈 결과이므로 다른 펭귄에서는 조금 달라질 수 있습니다. 8단계 혼동행렬의 대각선 칸을 더하면 맞힌 마리 수를 직접 확인할 수 있습니다.</details>

### 질문 2

**나무 깊이를 3에서 1로 줄이면 두 정확도는 어떻게 될까요?**

먼저 예측하고 m1으로 실험하세요. 원래 model과 pred는 유지합니다.

- 학습용과 평가용 모두 거의 그대로다.
- 학습용과 평가용 모두 크게 떨어진다.
- 학습용은 떨어지지만 평가용은 오른다.
- 학습용과 평가용 모두 1.0이 된다.

자유 실험 셀에서 확인할 코드:

```python
m1 = DecisionTreeClassifier(max_depth=1, random_state=42)
m1.fit(X_train, y_train)
print("학습용:", m1.score(X_train, y_train), "평가용:", m1.score(X_test, y_test))
```

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>기본 분할에서 깊이 1인 모델은 학습용 약 0.79, 평가용 약 0.78로 둘 다 낮아집니다. 한 번 나눈 두 잎으로 예측할 수 있는 종은 최대 두 종류이므로 세 종을 충분히 구분하기 어렵습니다. 내 실험 결과를 왼쪽 깊이 3 모델과 비교하세요. 나무를 더 깊게 하면 이번 데이터에서는 평가 정확도가 오르기도 합니다. 깊이의 적절함은 학습용 점수 하나로 판단하지 않습니다.</details>

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 08. 어떤 종을 헷갈렸는지 살펴보기

혼동행렬은 실제 종과 예측 종의 조합별 개수입니다. 행은 실제 정답, 열은 모델의 예측입니다. 대각선은 맞힌 개수, 대각선 밖은 헷갈린 개수입니다. 숫자 0도 포함해 모든 칸을 읽어 보세요.

**코드 완성 미션:** y_test와 pred로 혼동행렬 cm을 만들고 축 이름과 함께 그리세요.

<details><summary>작성 힌트</summary>첫 인자는 실제 정답 y_test, 두 번째는 예측값 pred입니다. 행과 열의 방향을 바꾸지 마세요.</details>

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(___, pred, labels=labels)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Penguin classifier: test records")
plt.xlabel("Predicted species")
plt.ylabel("Actual species")
plt.tight_layout()
plt.show()
print("평가 기록 수:", cm.sum())

### 질문 1

**실제 Gentoo를 Adelie로 예측한 기록은 어느 칸에 있나요?**

행은 실제 종, 열은 예측한 종입니다.

- Adelie 행, Gentoo 열
- Gentoo 행, Gentoo 열
- Gentoo 행, Adelie 열
- Adelie 열의 숫자를 모두 더한 값

내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>행(세로축)은 실제 종, 열(가로축)은 예측한 종입니다. 이 칸은 맨 아래 행의 왼쪽 칸입니다. 첫 번째 보기는 방향을 뒤집어 읽은 칸이라, 실제 Adelie를 Gentoo로 예측한 수를 나타냅니다. 기본 실행 결과에서는 두 칸의 숫자도 다르므로 직접 확인해 보세요.</details>

### 질문 2

**오분류 한 칸을 읽고, 산점도에서 본 모습과 비교해 보세요.**

‘실제 종 → 예측 종: 마릿수’로 적으세요. 오분류가 없다면 그 의미를 설명하세요.



내 답: 

<details><summary>답을 작성한 뒤 해설과 비교</summary>작성 예(한 번의 기본 실행 결과): 실제 Gentoo를 Adelie로 예측한 기록이 2마리였다. 산점도에서는 Gentoo가 대체로 따로 모였던 것과 달라서, 그 그림만으로 모델의 모든 판단을 설명하기는 어렵다. 내 숫자가 다르면 내 결과를 적습니다. 모델은 특성 4개, 산점도는 2개를 사용하므로 그림의 겹침이 오분류 원인이라고 단정하지 않습니다. 오분류가 없다면 대각선 밖이 모두 0임을 쓰고, 이번 평가 데이터에 한정된 결과임을 설명하세요.</details>

자기점검:
- [ ] 행(실제)과 열(예측)을 바른 순서로 읽었다.
- [ ] 칸의 숫자를 내 실행 결과에서 그대로 적었다.
- [ ] 산점도와 비교할 때 '원인이다'라고 단정하지 않고 비슷한지 다른지만 썼다.

**자유 실험:** 아래 셀에서 값을 바꾸거나 새 코드를 작성해 보세요. 앞에서 만든 변수를 함께 사용합니다.

## 내 말로 정리하기

1. 결측치와 잘못된 표기는 어떻게 다른가요? →
2. 산점도에서 본 특징과 모델의 분류 결과를 연결해 설명해 보세요. →
3. 평가용 정확도와 혼동행렬은 각각 무엇을 알려 주나요? →

**더 해보기:** 7단계의 max_depth를 바꾸고 7→8을 다시 실행해 비교하세요.
**생략한 과정:** 반복 countplot·boxplot·regplot·상관관계 히트맵·MinMax 정규화·나무 시각화.
원본 자료와 CSV는 reference 폴더에 보존합니다. 실제 수업 데이터의 출처는 원본 노트북에 기재된 Kaggle 링크를 따릅니다.
